In [1]:
from unike.module.model import RGCN, CompGCN
import sys

sys.path.extend(['.', '..'])

from q import link, drug_ent_indexs, indication_rel_index, dmd_ent_index, add_id

In [2]:
RGCN_model = RGCN(
	ent_tol = 121649,
	rel_tol = 22,
	dim = 200,
	num_layers = 2
)
RGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/RGCN_entrie_Accel_20250910-1000.pth")

In [3]:
CompGCN_model = CompGCN(
    ent_tol = 121649,
    rel_tol = 22,
    dim = 50
)
CompGCN_model.load_checkpoint("/home/wangtao/src/kg4rd/src/kg4rd/kge/checkpoints/A/CompGCN_entrie_Accel_20250910-1000.pth")

In [4]:
RGCN_result = add_id(link.link(
    drug_ent_indexs,
    [indication_rel_index],
    [dmd_ent_index],
    RGCN_model, 'cuda:0'
))

In [5]:
CompGCN_result = add_id(link.link(
    drug_ent_indexs,
    [indication_rel_index],
    [dmd_ent_index],
    CompGCN_model, 'cuda:0'
))

In [8]:
RGCN_result.to_csv("./RGCN.csv", index=False)
CompGCN_result.to_csv("./CompGCN.csv", index=False)

In [7]:
intersect_head = 100
RGCN_head_set = set(RGCN_result['head'].head(intersect_head).to_list())
CompGCN_head_set = set(CompGCN_result['head'].head(intersect_head).to_list())
intersect_set = RGCN_head_set & CompGCN_head_set
RGCN_result.query('head in @intersect_set')[['head', 'head_ent', 'head_id', 'uid']].to_csv(f"./intersect_result_{intersect_head}.csv", index=False)